# 01. ConversationBufferMemory / ConversationBufferWindowMemory → LangGraph Checkpointer

| legacy (`langchain_classic.memory`) | LangGraph |
|---|---|
| `ConversationBufferMemory()` | `InMemorySaver()` 를 `compile(checkpointer=...)` 에 연결 |
| `memory.save_context(inputs, outputs)` | 자동 저장 (그래프 실행마다) / 수동: `graph.update_state(config, {...})` |
| `memory.load_memory_variables({})` | `graph.get_state(config).values["messages"]` |
| `return_messages=False` (문자열 history) | `get_buffer_string(messages)` |
| 메모리 객체 1개 = 대화 1개 | `thread_id` 하나 = 대화 1개 (한 checkpointer가 여러 대화를 관리) |
| `ConversationBufferWindowMemory(k=2)` | `trim_messages(...)` (보내는 것만 자르기) 또는 `RemoveMessage` (실제로 삭제) |

> **핵심 변화**: 메모리를 "체인에 붙이는 객체"로 관리하던 방식에서, **그래프 상태(state)를 checkpointer가 thread 단위로 저장**하는 방식으로 바뀌었습니다.

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 1. 그래프 + checkpointer 만들기
`MessagesState`는 `messages` 리스트를 가진 상태이며, 새 메시지는 기존 리스트에 **추가(append)** 됩니다.

In [2]:
from langchain_core.messages import HumanMessage, AIMessage, get_buffer_string
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import InMemorySaver


def chatbot(state: MessagesState):
    # state["messages"] 에는 같은 thread 의 이전 대화가 모두 들어있다
    return {"messages": [llm.invoke(state["messages"])]}


builder = StateGraph(MessagesState)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

memory = InMemorySaver()  # ← ConversationBufferMemory 의 역할
graph = builder.compile(checkpointer=memory)

## 2. `save_context` 대응: `update_state`
LLM을 호출하지 않고 대화 내용을 직접 기록하고 싶을 때 사용합니다. `config`의 `thread_id`가 legacy의 "메모리 객체"를 대신합니다.

In [3]:
config = {"configurable": {"thread_id": "bank-1"}}

dialog = [
    ("안녕하세요, 비대면으로 은행 계좌를 개설하고 싶습니다. 어떻게 시작해야 하나요?",
     "안녕하세요! 계좌 개설을 원하신다니 기쁩니다. 먼저, 본인 인증을 위해 신분증을 준비해 주시겠어요?"),
    ("네, 신분증을 준비했습니다. 이제 무엇을 해야 하나요?",
     "감사합니다. 신분증 앞뒤를 명확하게 촬영하여 업로드해 주세요. 이후 본인 인증 절차를 진행하겠습니다."),
    ("사진을 업로드했습니다. 본인 인증은 어떻게 진행되나요?",
     "업로드해 주신 사진을 확인했습니다. 이제 휴대폰을 통한 본인 인증을 진행해 주세요. 문자로 발송된 인증번호를 입력해 주시면 됩니다."),
]

for human, ai in dialog:
    # legacy: memory.save_context(inputs={"human": human}, outputs={"ai": ai})
    graph.update_state(
        config,
        {"messages": [HumanMessage(human), AIMessage(ai)]},
        as_node="chatbot",
    )

## 3. `load_memory_variables` 대응: `get_state`

In [4]:
messages = graph.get_state(config).values["messages"]

# return_messages=True 와 같은 형태 (메시지 객체 리스트)
for m in messages:
    print(type(m).__name__, "|", m.content[:40])

HumanMessage | 안녕하세요, 비대면으로 은행 계좌를 개설하고 싶습니다. 어떻게 시작해야 
AIMessage | 안녕하세요! 계좌 개설을 원하신다니 기쁩니다. 먼저, 본인 인증을 위해 
HumanMessage | 네, 신분증을 준비했습니다. 이제 무엇을 해야 하나요?
AIMessage | 감사합니다. 신분증 앞뒤를 명확하게 촬영하여 업로드해 주세요. 이후 본인
HumanMessage | 사진을 업로드했습니다. 본인 인증은 어떻게 진행되나요?
AIMessage | 업로드해 주신 사진을 확인했습니다. 이제 휴대폰을 통한 본인 인증을 진행


In [5]:
# return_messages=False (기본값) 와 같은 형태 (문자열)
print(get_buffer_string(messages))

Human: 안녕하세요, 비대면으로 은행 계좌를 개설하고 싶습니다. 어떻게 시작해야 하나요?
AI: 안녕하세요! 계좌 개설을 원하신다니 기쁩니다. 먼저, 본인 인증을 위해 신분증을 준비해 주시겠어요?
Human: 네, 신분증을 준비했습니다. 이제 무엇을 해야 하나요?
AI: 감사합니다. 신분증 앞뒤를 명확하게 촬영하여 업로드해 주세요. 이후 본인 인증 절차를 진행하겠습니다.
Human: 사진을 업로드했습니다. 본인 인증은 어떻게 진행되나요?
AI: 업로드해 주신 사진을 확인했습니다. 이제 휴대폰을 통한 본인 인증을 진행해 주세요. 문자로 발송된 인증번호를 입력해 주시면 됩니다.


## 4. 실제 대화: 저장/불러오기가 자동
legacy에서는 `ConversationChain`이 호출 전후로 `load_memory_variables`/`save_context`를 대신 호출해 주었습니다. LangGraph에서는 **입력으로 새 메시지만** 넘기면 이전 대화는 checkpointer가 불러오고, 결과도 자동으로 저장합니다.

In [6]:
result = graph.invoke(
    {"messages": [HumanMessage("지금까지 제가 어떤 단계까지 진행했나요? 한 문장으로 알려주세요.")]},
    config,
)
print(result["messages"][-1].content)
print("저장된 메시지 수:", len(graph.get_state(config).values["messages"]))

현재 신분증 사진을 업로드하고 본인 인증을 위한 인증번호 입력 단계에 있습니다.
저장된 메시지 수: 8


In [7]:
# thread_id 가 다르면 완전히 별개의 대화 (legacy 에서 메모리 객체를 새로 만든 것과 같음)
other = {"configurable": {"thread_id": "bank-2"}}
result = graph.invoke({"messages": [HumanMessage("지금까지 제가 어떤 단계까지 진행했나요? 한 문장으로 알려주세요.")]}, other)
print(result["messages"][-1].content)

죄송하지만, 제가 당신의 진행 상황에 대한 구체적인 정보를 알 수 없습니다. 어떤 프로젝트나 작업에 대해 말씀해 주시면 도움을 드릴 수 있습니다.


## 5. 가장 짧은 방법: `create_agent` + checkpointer
legacy 경고 메시지(`Use langchain.agents.create_agent instead ...`)가 권장하는 방법입니다. 그래프를 직접 만들지 않아도 됩니다. (`ConversationChain` 의 대체재)

In [8]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="You are a helpful assistant.",
    checkpointer=InMemorySaver(),
)

cfg = {"configurable": {"thread_id": "agent-1"}}
agent.invoke({"messages": [{"role": "user", "content": "안녕, 내 이름은 테디야."}]}, cfg)
answer = agent.invoke({"messages": [{"role": "user", "content": "내 이름이 뭐라고 했지?"}]}, cfg)
print(answer["messages"][-1].content)

너의 이름은 테디야. 맞지?


## 6. `ConversationBufferWindowMemory(k=2)` 대응
전용 클래스는 없고, 두 가지 방법 중 선택합니다.

* **방법 A – `trim_messages`**: 상태에는 전체 대화를 두고, **LLM에 보낼 때만** 최근 k턴으로 자릅니다. (legacy Window 메모리와 동일한 동작)
* **방법 B – `RemoveMessage`**: 오래된 메시지를 상태에서 **실제로 삭제**합니다. (저장 공간 절약)

In [9]:
from langchain_core.messages import trim_messages

K = 2  # 최근 2턴(= Human/AI 4개 메시지)

# legacy: ConversationBufferWindowMemory(k=2).load_memory_variables({})["history"]
window = trim_messages(
    messages,               # 위에서 저장한 3턴(6개) 대화
    token_counter=len,      # 토큰 대신 "메시지 개수"로 센다
    max_tokens=K * 2,
    strategy="last",        # 최근 것부터 남김
    start_on="human",       # 잘린 결과가 Human 메시지로 시작하도록
)
for m in window:
    print(type(m).__name__, "|", m.content[:40])

HumanMessage | 네, 신분증을 준비했습니다. 이제 무엇을 해야 하나요?
AIMessage | 감사합니다. 신분증 앞뒤를 명확하게 촬영하여 업로드해 주세요. 이후 본인
HumanMessage | 사진을 업로드했습니다. 본인 인증은 어떻게 진행되나요?
AIMessage | 업로드해 주신 사진을 확인했습니다. 이제 휴대폰을 통한 본인 인증을 진행


In [10]:
# 방법 A: 노드 안에서 LLM에 보낼 때만 잘라 쓰기 (state 는 전체 보존)
def window_chatbot(state: MessagesState):
    recent = trim_messages(
        state["messages"], token_counter=len, max_tokens=K * 2 + 1,  # +1 = 이번 질문
        strategy="last", start_on="human",
    )
    return {"messages": [llm.invoke(recent)]}


window_graph = (
    StateGraph(MessagesState)
    .add_node("chatbot", window_chatbot)
    .add_edge(START, "chatbot")
    .compile(checkpointer=InMemorySaver())
)

In [11]:
# 방법 B: 응답 후 오래된 메시지를 상태에서 삭제
from langchain_core.messages import RemoveMessage


def delete_old_messages(state: MessagesState):
    old = state["messages"][: -K * 2]
    return {"messages": [RemoveMessage(id=m.id) for m in old]}


delete_graph = (
    StateGraph(MessagesState)
    .add_node("chatbot", chatbot)
    .add_node("delete_old_messages", delete_old_messages)
    .add_edge(START, "chatbot")
    .add_edge("chatbot", "delete_old_messages")
    .compile(checkpointer=InMemorySaver())
)

questions = ["내 이름은 테디야.", "나는 서울에 살아.", "나는 개발자야.", "내 이름이 뭐지? 모르면 모른다고 해."]
for g, name in [(window_graph, "방법 A (trim)"), (delete_graph, "방법 B (delete)")]:
    cfg = {"configurable": {"thread_id": "window"}}
    for q in questions:
        out = g.invoke({"messages": [HumanMessage(q)]}, cfg)
    print(f"[{name}] 마지막 답변: {out['messages'][-1].content}")
    print(f"[{name}] state 에 남은 메시지 수: {len(g.get_state(cfg).values['messages'])}\n")

[방법 A (trim)] 마지막 답변: 죄송하지만, 당신의 이름은 알 수 없습니다. 이름을 알려주시면 좋겠네요!
[방법 A (trim)] state 에 남은 메시지 수: 8



[방법 B (delete)] 마지막 답변: 죄송하지만, 당신의 이름은 알 수 없습니다. 이름을 알려주시면 좋겠네요!
[방법 B (delete)] state 에 남은 메시지 수: 4



### 정리
* 두 방법 모두 최근 2턴만 LLM에 전달되므로 첫 턴에서 말한 이름을 기억하지 못합니다. (Window 메모리의 의도된 동작)
* 방법 A는 state 에 8개가 모두 남고, 방법 B는 4개만 남습니다.